In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import math
import re

from shapely.geometry import Point, LineString
from shapely import wkt
from shapely.ops import unary_union
from data2dd_funcs import (wrapdd, data2dd)

In [ ]:
# TODO:s
# Add new technology for land-based HVDC lines. Currently using costs and technical data from HVDC Marine. 

In [ ]:
euroshape = snakemake.input.euroshape

In [ ]:
europe = (
    gpd
    .read_file(euroshape)
    #.query("CNTR_CODE == @aggregated_regions")
    .set_index("index")
)

In [ ]:
regions = europe.index.to_list()

In [ ]:
countries = europe.CNTR_CODE.unique().tolist()

In [ ]:
# This is set in the config and is an output from a preprocessing step. 
transmission_alpha = snakemake.params.transmission_alpha

## Load transmission data

In [ ]:
# Load the AC transmission lines (10.1038/s41597-025-04550-7)
lines = pd.read_csv(snakemake.input.existing_lines, quotechar="'")
lines["geometry"] = lines["geometry"].apply(wkt.loads)
lines = gpd.GeoDataFrame(lines, geometry="geometry", crs="EPSG:4326")

In [ ]:
# DC links. Do not need processing and will be added towards the end. 
links = pd.read_csv(snakemake.input.existing_links, quotechar="'")
links["geometry"] = links["geometry"].apply(wkt.loads)
links = gpd.GeoDataFrame(links, geometry="geometry", crs="EPSG:4326")

In [ ]:
# Loop through all lines in the dataset and check which of the zones in our shapefile that the lines go between. 
# We only save those that go between different zones, meaning that we drop everything within e.g. a NUTS2 region. 
zone_1, zone_2 = [], []
s_nom = []
line = []
for i in range(len(lines)):
    z_1 = europe[europe.intersects(list(lines.geometry.loc[i].boundary.geoms)[0].buffer(1e-3))]
    z_2 = europe[europe.intersects(list(lines.geometry.loc[i].boundary.geoms)[1].buffer(1e-3))]

    if not z_1.empty and not z_2.empty and z_1.index[0] != z_2.index[0]:
        zone = [z_1.index[0], z_2.index[0]]
        zone.sort()
        zone_1.append(zone[0])
        zone_2.append(zone[1])
        s_nom.append(lines.loc[i].s_nom)
        line.append(lines.loc[[i]].geometry.iloc[0])

existing_lines = gpd.GeoDataFrame(pd.DataFrame({'Zone1':zone_1, 'Zone2':zone_2, 's_nom':s_nom, 'geometry':line})).set_crs(europe.crs)
existing_lines_grouped = existing_lines[['Zone1', 'Zone2', 's_nom']].groupby(['Zone1', 'Zone2']).sum().reset_index()

In [ ]:
existing_lines.head()

In [ ]:
# Plot the lines to get an idea of what the data looks like
fig, ax = plt.subplots(figsize=(12,10))

europe.boundary.plot(ax=ax,linewidth=1,color='lightgray')
#europe.dissolve(by='CNTR_CODE').boundary.plot(ax=ax,linewidth=1,color='green')
existing_lines.plot(ax=ax,linewidth=1,color='teal',linestyle='--',alpha=0.5)

## Check for void zones

In [ ]:
# The following is a check to ensure that there are no zones that are isolated due to the transmission data. 
# This could happen if there would only be distribution grid lines into a zone, which are not included in our dataset.
intersections = gpd.sjoin(
    lines,
    europe[['CNTR_CODE', 'geometry']],
    how='left',
    predicate='intersects'
)

In [ ]:
intersections['has_line'] = ~intersections['line_id'].isna()

zone_has_line = (
    intersections
    .groupby(['index', 'CNTR_CODE'], as_index=False)['has_line']
    .max()   # True if any intersecting line, False if none
)

# Mark void zones (no high-voltage lines)
zone_has_line['is_void'] = ~zone_has_line['has_line']


In [ ]:
country_stats = (
    zone_has_line
    .groupby('CNTR_CODE')
    .agg(
        total_zones=('index', 'count'),
        void_zones=('is_void', 'sum')
    )
    .reset_index()
)
country_stats['void_pct'] = 100 * country_stats['void_zones'] / country_stats['total_zones']

In [ ]:
for country in country_stats.CNTR_CODE:
    void_zones = country_stats.query("CNTR_CODE == @country").void_zones.sum()
    if void_zones != 0:
        print(f'Warning: {country} has {void_zones} regions not touched by the high-voltage grid')

## Data processing

In [ ]:
# Next we process the data and go from s_nom to line capacity using the transmission_alpha variable. 
ac_lines_scaled = existing_lines_grouped.assign(links_cap_exist_MW = lambda x : x.s_nom*transmission_alpha)

In [ ]:
europe2 = europe.to_crs('epsg:25833')

In [ ]:
distance = []
zone_1 = []
zone_2 = []
# Get the distances between centroids. EPSG:25833 is in meters
for i in ac_lines_scaled[['Zone1', 'Zone2']].values:
    distance.append(europe2.centroid.loc[i[0]].distance(europe2.centroid.loc[i[1]]))
    zone_1.append(i[0])
    zone_2.append(i[1])

In [ ]:
ac_lines_scaled = ac_lines_scaled.groupby(['Zone1', 'Zone2']).sum()

In [ ]:
# Convert to kilometers
ac_lines_scaled['links_dist'] = pd.DataFrame({'Zone1':zone_1, 'Zone2':zone_2, 'distance':distance}).groupby(['Zone1', 'Zone2']).sum().distance/1000

In [ ]:
ac_lines_scaled = ac_lines_scaled.reset_index().assign(Tech = 'HVAC_OHL').loc[:,["Zone1", "Zone2", "Tech", "links_cap_exist_MW", "links_dist"]]

In [ ]:
ac_lines_scaled.head(10)

## Compare with known dataset

In [ ]:
# Now we want to ensure that our approximations using the transmission_alpha does not over/underestimate the line capacity in cases where we have more accurate data
# To do this, we extract the country codes for our lines, compute a scaling factor and apply it to our detailed dataset.
df_existing = pd.read_csv(snakemake.input.existing_transmission,skiprows=1).query("Zone1 == @countries and Zone2 == @countries")

In [ ]:
df_existing.head()

In [ ]:
def normalize_pair(df):
    a = df[["Zone1", "Zone2"]].min(axis=1)
    b = df[["Zone1", "Zone2"]].max(axis=1)
    df = df.copy()
    df["Zone1_norm"] = a
    df["Zone2_norm"] = b
    return df

trans_norm = normalize_pair(ac_lines_scaled)
exist_norm = normalize_pair(df_existing[["Zone1", "Zone2", "Tech"]])

merged = trans_norm.merge(
    exist_norm[["Zone1_norm", "Zone2_norm", "Tech"]],
    on=["Zone1_norm", "Zone2_norm", "Tech"],
    how="left",
    indicator=True
)

new_rows = merged[merged["_merge"] == "left_only"][
    ["Zone1", "Zone2", "Tech", "links_cap_exist_MW", "links_dist"]
]

df_updated = pd.concat([df_existing, new_rows], ignore_index=True)
df_updated = round(df_updated,0)

In [ ]:
zone_to_country = europe["CNTR_CODE"]

df_lines = df_updated.copy()
df_lines["country1"] = df_lines["Zone1"].map(zone_to_country).fillna(df_lines["Zone1"])
df_lines["country2"] = df_lines["Zone2"].map(zone_to_country).fillna(df_lines["Zone2"])

df_lines[["cA", "cB"]] = np.sort(
    df_lines[["country1", "country2"]].values, axis=1
)

In [ ]:
countries = set(europe["CNTR_CODE"].unique())

is_country_level = df_lines["Zone1"].isin(countries) & df_lines["Zone2"].isin(countries)

df_country = df_lines[is_country_level].copy()
df_detailed = df_lines[~is_country_level].copy()

In [ ]:
country_caps = (
    df_country
    .groupby(["cA", "cB"], as_index=False)["links_cap_exist_MW"]
    .sum()
    .rename(columns={"links_cap_exist_MW": "country_cap"})
)

detailed_caps = (
    df_detailed
    .groupby(["cA", "cB"], as_index=False)["links_cap_exist_MW"]
    .sum()
    .rename(columns={"links_cap_exist_MW": "detailed_cap"})
)

scaling = pd.merge(country_caps, detailed_caps, on=["cA", "cB"], how="inner")

# Only compute factor where detailed_cap > 0
scaling["scale"] = scaling["country_cap"] / scaling["detailed_cap"]

In [ ]:
df_exist_lines_scaled = df_detailed.merge(
    scaling[["cA", "cB", "scale"]],
    on=["cA", "cB"],
    how="left"
)

df_exist_lines_scaled["links_cap_exist_MW"] = round((
    df_exist_lines_scaled["links_cap_exist_MW"]
    * df_exist_lines_scaled["scale"].fillna(1.0)
),0)

In [ ]:
# Drop helper columns
df_exist_lines_scaled = df_exist_lines_scaled.drop(columns=["country1", "country2", "cA", "cB", "scale"])

In [ ]:
df_exist_lines_scaled.head()

## Add in DC links

In [ ]:
# Similar process as with the AC lines, but we do not need to convert from s_nom. 
zone_1, zone_2 = [], []
s_nom = []
line = []

for i in range(len(links)):
    z_1 = europe[europe.intersects(list(links.geometry.loc[i].boundary.geoms)[0].buffer(1e-3))]
    z_2 = europe[europe.intersects(list(links.geometry.loc[i].boundary.geoms)[1].buffer(1e-3))]

    if not z_1.empty and not z_2.empty and z_1.index[0] != z_2.index[0]:
        zone = [z_1.index[0], z_2.index[0]]
        zone.sort()
        zone_1.append(zone[0])
        zone_2.append(zone[1])
        s_nom.append(links.loc[i].p_nom)
        line.append(links.loc[[i]].geometry.iloc[0])

dc_links = gpd.GeoDataFrame(pd.DataFrame({'Zone1':zone_1, 'Zone2':zone_2, 's_nom':s_nom, 'geometry':line})).set_crs(europe.crs)
dc_links_grouped = dc_links[['Zone1', 'Zone2', 's_nom']].groupby(['Zone1', 'Zone2']).sum().reset_index()

In [ ]:
existing_dc = dc_links_grouped.rename(columns={'s_nom' : 'links_cap_exist_MW'}).copy()

In [ ]:
distance = []
zone_1 = []
zone_2 = []
for i in existing_dc[['Zone1', 'Zone2']].values:
    distance.append(europe2.centroid.loc[i[0]].distance(europe2.centroid.loc[i[1]]))
    zone_1.append(i[0])
    zone_2.append(i[1])

In [ ]:
existing_dc = existing_dc.groupby(['Zone1', 'Zone2']).sum()
existing_dc['links_dist'] = round(pd.DataFrame({'Zone1':zone_1, 'Zone2':zone_2, 'distance':distance}).groupby(['Zone1', 'Zone2']).sum().distance/1000,0)

In [ ]:
existing_dc = existing_dc.reset_index().assign(Tech = 'HVDC_MarineIC').loc[:,["Zone1", "Zone2", "Tech", "links_cap_exist_MW", "links_dist"]]

In [ ]:
existing_dc.head()

In [ ]:
df_exist_lines_scaled = pd.concat([df_exist_lines_scaled,existing_dc])
df_exist_lines_scaled['links_cap_exist_MW'] = df_exist_lines_scaled['links_cap_exist_MW'].astype(float)

## Add planned lines

In [ ]:
# TODO: Include an option to select a specific project_status (e.g. in_permitting or under_construction)

In [ ]:
def make_linestring(row):
    return LineString([(row["x0"], row["y0"]), (row["x1"], row["y1"])])

In [ ]:
new_lines = pd.read_csv(snakemake.input.new_lines).assign(new_upgrade = 'new')
upg_lines = pd.read_csv(snakemake.input.upg_lines).assign(new_upgrade = 'upgrade')

In [ ]:
# We need to extract the Project ID from the TYNDP in order to match it with the Transfer capacity increase
# Some projects contain multiple lines and are merged. 

new_lines['Project_ID'] = new_lines['tags'].str.extract(r'tyndp2020_proj_id"=>"(\d+)"')
new_lines['Project_ID'] = new_lines['Project_ID'].astype(int)
new_lines = gpd.GeoDataFrame(new_lines, geometry=new_lines.apply(lambda r: LineString([(r['x0'], r['y0']), (r['x1'], r['y1'])]),axis=1),crs='EPSG:4326')
new_lines = new_lines.groupby(['Project_ID',], as_index=False).agg({'geometry': lambda geoms: unary_union(geoms),'project_status': 'first','new_upgrade' : 'first'})

upg_lines['Project_ID'] = upg_lines['tags'].str.extract(r'tyndp2020_proj_id"=>"(\d+)"')
upg_lines['Project_ID'] = upg_lines['Project_ID'].astype(int)
upg_lines = gpd.GeoDataFrame(upg_lines, geometry=upg_lines.apply(lambda r: LineString([(r['x0'], r['y0']), (r['x1'], r['y1'])]),axis=1),crs='EPSG:4326')
upg_lines = upg_lines.groupby(['Project_ID',], as_index=False).agg({'geometry': lambda geoms: unary_union(geoms),'project_status': 'first','new_upgrade' : 'first'})

In [ ]:
# TODO: See if we can use the updated TYNDP from 2024. Currently there are some issues which prevents this. 
# For example, project ID 33 includes two lines with different capacities. 
tyndp2024 = pd.read_csv(snakemake.input.tyndp2024,skiprows=1).loc[:,['Project ID','Country','Transfer capacity increase A-B (MW)','Transfer capacity increase B-A (MW)']].rename(columns={'Project ID': 'Project_ID'})
tyndp2024['TransferCapacity_TYNDP2024'] = tyndp2024[
    ['Transfer capacity increase A-B (MW)',
     'Transfer capacity increase B-A (MW)']
].max(axis=1)
tyndp2020 = pd.read_csv(snakemake.input.tyndp2020,skiprows=1).loc[:,['Project ID','Country','Transfer capacity increase A-B (MW)','Transfer capacity increase B-A (MW)']].rename(columns={'Project ID': 'Project_ID'})
tyndp2020['TransferCapacity_TYNDP2020'] = tyndp2020[
    ['Transfer capacity increase A-B (MW)',
     'Transfer capacity increase B-A (MW)']
].max(axis=1)

In [ ]:
tyndp2020 = tyndp2020.set_index('Project_ID')

In [ ]:
tyndp2024 = tyndp2024.set_index('Project_ID')

In [ ]:
tyndp = (
    tyndp2024
    .merge(
        tyndp2020[["TransferCapacity_TYNDP2020"]],
        left_index=True,
        right_index=True,
        how="outer",
    )
    .assign(
        TransferCapacity=lambda df: df["TransferCapacity_TYNDP2024"]
        .combine_first(df["TransferCapacity_TYNDP2020"])
    )
).reset_index()


In [ ]:
planned_lines = pd.concat([new_lines,upg_lines]).reset_index(drop=True)
planned_lines = gpd.GeoDataFrame(planned_lines, geometry="geometry", crs='EPSG:4326')

planned_lines = planned_lines.merge(
    tyndp[['Project_ID', 'TransferCapacity']],
    on='Project_ID',
    how='left' 
)

s = planned_lines['TransferCapacity']

# TODO: Fix the multiple rows
# The dataset is a not cleaned and some rows contain multiple rows, no data or a dash. We ignore these for now.
mask_bad = (
    s.isna()  # None / NaN
    | s.astype(str).str.contains(r'\n|^-$', na=False)  # newline OR exactly '-'
)

dropped_lines = planned_lines[mask_bad].reset_index(drop=True)

planned_lines = planned_lines[~mask_bad]
# Ignore links where the capacity addition is 0 for now. 
planned_lines = planned_lines.query("TransferCapacity != '0'").reset_index(drop=True)

planned_lines = planned_lines[
    (planned_lines.geometry.geom_type == "LineString")
    & (~planned_lines.geometry.is_empty)
    & (planned_lines.geometry.length > 0)
].reset_index(drop=True)

In [ ]:
planned_lines['TransferCapacity'] = planned_lines['TransferCapacity'].astype(float)

In [ ]:
zone_1, zone_2 = [], []
s_nom = []
line = []

for i in range(len(planned_lines)):
    z_1 = europe[europe.intersects(list(planned_lines.geometry.loc[i].boundary.geoms)[0].buffer(1e-3))]
    z_2 = europe[europe.intersects(list(planned_lines.geometry.loc[i].boundary.geoms)[1].buffer(1e-3))]

    if not z_1.empty and not z_2.empty and z_1.index[0] != z_2.index[0]:
        zone = [z_1.index[0], z_2.index[0]]
        zone.sort()
        zone_1.append(zone[0])
        zone_2.append(zone[1])
        s_nom.append(planned_lines.loc[i].TransferCapacity)
        line.append(planned_lines.loc[[i]].geometry.iloc[0])

planned_relevant_lines = gpd.GeoDataFrame(pd.DataFrame({'Zone1':zone_1, 'Zone2':zone_2, 's_nom':s_nom, 'geometry':line})).set_crs(europe.crs)
planned_relevant_lines_grouped = planned_relevant_lines[['Zone1', 'Zone2', 's_nom']].groupby(['Zone1', 'Zone2']).sum().reset_index().rename(columns={'s_nom' : 'links_cap_planned_MW'})

distance = []
zone_1 = []
zone_2 = []
for i in planned_relevant_lines_grouped[['Zone1', 'Zone2']].values:
    distance.append(europe2.centroid.loc[i[0]].distance(europe2.centroid.loc[i[1]]))
    zone_1.append(i[0])
    zone_2.append(i[1])

planned_relevant_lines_grouped = planned_relevant_lines_grouped.groupby(['Zone1', 'Zone2']).sum()

In [ ]:
planned_relevant_lines_grouped['links_dist'] = round(pd.DataFrame({'Zone1':zone_1, 'Zone2':zone_2, 'distance':distance}).groupby(['Zone1', 'Zone2']).sum().distance/1000,0)

In [ ]:
planned_relevant_lines_grouped = planned_relevant_lines_grouped.reset_index().assign(Tech = 'HVAC_OHL').loc[:,["Zone1", "Zone2", "Tech", "links_cap_planned_MW", "links_dist"]]

In [ ]:
zone_1, zone_2 = [], []
s_nom = []
line = []
project_ids = []   # <- new

for i, geom in dropped_lines.geometry.items():
    # Skip empty or missing geometries
    if geom is None or geom.is_empty:
        continue

    boundary_geoms = list(geom.boundary.geoms)

    # Skip if you don't have both endpoints
    if len(boundary_geoms) < 2:
        continue

    g1 = boundary_geoms[0].buffer(1e-3)
    g2 = boundary_geoms[1].buffer(1e-3)

    z_1 = europe[europe.intersects(g1)]
    z_2 = europe[europe.intersects(g2)]

    if not z_1.empty and not z_2.empty:
        zone = sorted([z_1.index[0], z_2.index[0]])
        zone_1.append(zone[0])
        zone_2.append(zone[1])
        s_nom.append(dropped_lines.loc[i, "TransferCapacity"])
        line.append(geom)
        project_ids.append(dropped_lines.loc[i, "Project_ID"])  # <- keep ID

gdf_dropped = gpd.GeoDataFrame(
    {
        "Project_ID": project_ids,
        "Zone1": zone_1,
        "Zone2": zone_2,
        "TransferCapacity": s_nom,
        "geometry": line,
    },
    crs=europe.crs,
)

In [ ]:
gdf_dropped

In [ ]:
gdf_dropped.to_csv(snakemake.output.droppedLines)

## Add planned links

In [ ]:
upg_links = pd.read_csv(snakemake.input.upg_links).assign(new_upgrade = 'upgrade')
new_links = pd.read_csv(snakemake.input.new_links).assign(new_upgrade = 'new')

In [ ]:
planned_links = pd.concat([new_links,upg_links]).reset_index(drop=True)

In [ ]:
planned_links.head()

In [ ]:
planned_links["geometry"] = planned_links.apply(make_linestring, axis=1)
planned_links = gpd.GeoDataFrame(planned_links, geometry="geometry", crs="EPSG:4326")  # WGS84 lon/lat

zone_1, zone_2 = [], []
s_nom = []
line = []

for i in range(len(planned_links)):
    z_1 = europe[europe.intersects(list(planned_links.geometry.loc[i].boundary.geoms)[0].buffer(1e-3))]
    z_2 = europe[europe.intersects(list(planned_links.geometry.loc[i].boundary.geoms)[1].buffer(1e-3))]

    if not z_1.empty and not z_2.empty and z_1.index[0] != z_2.index[0]:
        zone = [z_1.index[0], z_2.index[0]]
        zone.sort()
        zone_1.append(zone[0])
        zone_2.append(zone[1])
        s_nom.append(planned_links.loc[i].p_nom)
        line.append(planned_links.loc[[i]].geometry.iloc[0])

planned_relevant_links = gpd.GeoDataFrame(pd.DataFrame({'Zone1':zone_1, 'Zone2':zone_2, 's_nom':s_nom, 'geometry':line})).set_crs(europe.crs)
planned_relevant_links_grouped = planned_relevant_links[['Zone1', 'Zone2', 's_nom']].groupby(['Zone1', 'Zone2']).sum().reset_index().rename(columns={'s_nom' : 'links_cap_planned_MW'})

distance = []
zone_1 = []
zone_2 = []
for i in planned_relevant_links_grouped[['Zone1', 'Zone2']].values:
    distance.append(europe2.centroid.loc[i[0]].distance(europe2.centroid.loc[i[1]]))
    zone_1.append(i[0])
    zone_2.append(i[1])

planned_relevant_links_grouped = planned_relevant_links_grouped.groupby(['Zone1', 'Zone2']).sum()

In [ ]:
planned_relevant_links_grouped['links_dist'] = round(pd.DataFrame({'Zone1':zone_1, 'Zone2':zone_2, 'distance':distance}).groupby(['Zone1', 'Zone2']).sum().distance/1000,0)

In [ ]:
planned_relevant_links_grouped = planned_relevant_links_grouped.reset_index().assign(Tech = 'HVDC_MarineIC').loc[:,["Zone1", "Zone2", "Tech", "links_cap_planned_MW", "links_dist"]]

In [ ]:
planned_lines_links = pd.concat([planned_relevant_links_grouped,planned_relevant_lines_grouped])
planned_lines_links['links_cap_planned_MW'] = (
    planned_lines_links['links_cap_planned_MW'].astype(float)
)

In [ ]:
planned_lines_links


* How do we think about new transmission conceptually. 
	* Do we want to set upper limits between countries and then let the model decide how to allocate this?
	* Do we want to set relative upper limits (e.g. allow for 25% higher capacity)
	* Total network size
	* Fixed to existing
	* Freely expandable
	* Any other approach?


## Write existing to file

In [ ]:
df_exist_lines_scaled.head()

In [ ]:
planned_lines_links.head()

In [ ]:
trans_link_cap = (
    pd
    .concat([df_exist_lines_scaled, planned_lines_links])
    .fillna(0)
    .groupby(['Zone1','Zone2','Tech'],as_index=False).sum()
    .assign(combined_cap = lambda x : x.links_cap_exist_MW + x.links_cap_planned_MW)
    .rename(columns={'links_cap_exist_MW' : 'links_cap', 'links_cap_planned_MW' : 'links_lim_cap'})
)
trans_link_cap.query("links_cap != 0 & links_lim_cap != 0")

In [ ]:
tech_type = "trans"

links_out = np.array(
        trans_link_cap["Zone1"]
        + "."
        + trans_link_cap["Zone2"]
        + "."
        + trans_link_cap["Tech"]
    )

links_tech = pd.unique(trans_link_cap["Tech"])

set_outdd = []
param_outdd = []

set_outdd.append(wrapdd(data2dd(links_tech, []), "trans", "set"))
set_outdd.append(wrapdd(data2dd(links_out, []), "trans_links", "set"))

out_data = ["links_dist", "links_cap", "links_lim_cap"]

for od in out_data:
        out_par = tech_type + "_" + od

        param_outdd.append(
            wrapdd(data2dd(trans_link_cap[od].values, [links_out]), out_par, "parameter")
        )

In [ ]:
params = pd.read_csv(snakemake.input.transmission_params, skiprows=1)

nans = params.isnull()
params = params.where(~nans, other=0.0)

out_data = ["loss", "line_capex", "sub_capex", "varom"]

params = params[params["Technology Name (highRES)"].isin(links_tech)]

for od in out_data:
    out_par = tech_type + "_" + od

    param_outdd.append(
        wrapdd(
            data2dd(params[od], [params["Technology Name (highRES)"]]),
            out_par,
            "parameter",
        )
    )

param_outdd = np.concatenate(param_outdd, axis=0)
set_outdd = np.concatenate(set_outdd, axis=0)

pad = np.repeat(np.array(""), set_outdd.shape[0]).reshape(set_outdd.shape[0], 1)

outdd = np.concatenate((np.hstack((set_outdd, pad)), param_outdd), axis=0)

np.savetxt(snakemake.output.transdd, outdd, delimiter=" ", fmt="%s")